[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/04_mpc.ipynb)

# Part 4 — MPC-style planning

Model Predictive Control comes from process control. At each step you plan a sequence of the next *N* actions against a **forward model** — a cheap approximation of how the system responds — then execute only the first action, observe what actually happened, and re-plan from there. The rest of the plan is discarded every step.

The agentic version keeps that loop and changes who proposes. The LLM generates candidate actions from context; the forward model scores them. The LLM never decides which candidate is best.

**What does this notebook do?** We find the launch angle that maximises the range of a projectile with air resistance, treating each simulation as expensive. The LLM proposes angles worth trying, a quadratic surrogate fitted to what we have already measured predicts how good each one is, and we spend a real simulation only on the best of them. Of course, solving a ballistic ODE is **not** expensive - this is just a toy placeholder which might be swapped out for e.g. an expensive PDE solver.

This problem is memoryless — each launch is independent of the last — so there is no sequence of actions to plan, and we use the one-step form of the loop: propose several candidates, score them, commit to one. Section 4.4 covers when the full *N*-step version earns its cost.

**You are done when** the loop settles near 38°, below the 45° that would be optimal without drag.

## 4.0 API Setup

Same setup as Part 0, plus `numpy` and `scipy.integrate` for the simulator. This one also defines `llm_text`.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json, re
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0"

from google import genai
from google.genai import types as gtypes
import numpy as np
from scipy.integrate import odeint

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)


def llm_text(system_prompt, user_prompt, temperature=0.2):
    resp = generate_with_retry(
        contents=user_prompt,
        config=gtypes.GenerateContentConfig(
            system_instruction=system_prompt, temperature=temperature))
    return resp.text or ""

print(f"Gemini client ready (model={MODEL}).")

## 4.1 The environment

We consider a projectile experiencing a quadratic drag law. The agent will only ever see a black box oracle of the form `evaluate(angle) -> range`; it does not get to look at the ODE.

$$\ddot{x} = -c_d |v|\dot{x}, \qquad \ddot{y} = -g - c_d|v|\dot{y}$$

with $v_0 = 50$ m/s, $c_d = 0.01$ m$^{-1}$, $g = 9.81$ m/s².

Without drag the optimum is exactly 45°. With drag it drops to roughly 38°. That gap is what makes this a reasonable test: 45° is the standard textbook result and the obvious first guess, so the surrogate has to pull the search away from it.

In [ ]:
V0, CD, G = 50.0, 0.01, 9.81

def _rhs_drag(s, t):
    x, y, vx, vy = s
    v = np.sqrt(vx*vx + vy*vy)
    return [vx, vy, -CD*v*vx, -G - CD*v*vy]

def simulate_range(angle_deg):
    """Horizontal range (m) for a launch angle. This is the expensive action."""
    a = np.deg2rad(angle_deg)
    s0 = [0.0, 0.0, V0*np.cos(a), V0*np.sin(a)]
    t = np.linspace(0, 20, 4001)
    sol = odeint(_rhs_drag, s0, t)
    y = sol[:, 1]
    hit = np.where((y[:-1] >= 0) & (y[1:] < 0))[0]
    if len(hit) == 0:
        return float('nan')
    i = hit[0]
    frac = y[i] / (y[i] - y[i+1])
    return float(sol[i, 0] + frac * (sol[i+1, 0] - sol[i, 0]))

for a in (20.0, 45.0, 70.0):
    print(f"  {a:4.1f} deg -> {simulate_range(a):7.3f} m")

## 4.2 The forward model

MPC needs a forward model that is cheaper than acting. If we "simulated" a candidate angle by calling `simulate_range` on it, we would have paid the full cost of the action already, and the lookahead would buy nothing — it would be a grid search with extra steps.

So the forward model has to be a genuine **surrogate**: an approximation fitted to what we have already measured. Here that is a quadratic least-squares fit through the best few observations. It costs nothing to evaluate, and it only has to be accurate near the peak, which is the only region we care about.

In [ ]:
def fit_surrogate(obs, k=5):
    """Quadratic fit through the k best observations. This is the cheap forward model."""
    if len(obs) < 3:
        return None
    best = sorted(obs, key=lambda o: -o[1])[:max(k, 3)]
    a = np.array([o[0] for o in best], float)
    r = np.array([o[1] for o in best], float)
    if len(np.unique(a)) < 3:
        return None
    return np.polyfit(a, r, 2)

def surrogate_predict(coef, angle):
    return float(np.polyval(coef, angle))

# Three points are enough to locate the peak roughly.
demo = [(a, simulate_range(a)) for a in (20.0, 45.0, 70.0)]
coef = fit_surrogate(demo)
A, B, _ = coef
print(f"surrogate peak at {-B / (2 * A):.2f} deg")

## 4.3 The MPC loop

Each pass through the loop does four things: fit the surrogate to everything measured so far, ask the proposer for candidate angles, score every candidate with the surrogate, and spend one real `simulate_range` call on the winner.

The proposer never sees the scores and never picks. It supplies candidates; the surrogate ranks them. That split is what makes the pattern worth using — the LLM is good at suggesting plausible values from context, and a fitted quadratic is better at saying which one is highest.

`fit_surrogate` returns `None` until three distinct angles have been measured, so the first pass falls back to the first candidate.

In [ ]:
def mpc_optimize(propose, evaluate, budget=10, n_candidates=5,
                 seeds=(20.0, 45.0, 70.0)):
    """Plan -> simulate (surrogate) -> act (ONE real evaluation) -> re-plan."""
    obs = [(a, evaluate(a)) for a in seeds]
    while len(obs) < budget:
        coef = fit_surrogate(obs)                                # 1. world model
        cands = [c for c in propose(obs, n_candidates) if 0 < c < 90]     # 2. plan
        if not cands:
            break
        # fit_surrogate returns None until there are 3 distinct angles
        if coef is None:
            best = cands[0]
        else:
            best = max(cands, key=lambda a: surrogate_predict(coef, a))   # 3. simulate
        obs.append((best, evaluate(best)))                       # 4. act: one only
        print(f"  n={len(obs):2d}  acted on {best:6.2f} -> {obs[-1][1]:7.3f}")
    return obs

`llm_proposer` is the plan step. It shows the model everything measured so far and asks for new angles as a JSON list. The parse is deliberately forgiving — it pulls any numbers out of the reply — because a model told to emit "only a JSON list" will sometimes add a sentence anyway.

With a budget of 10 and three seed angles, this makes seven LLM calls.

In [ ]:
def llm_proposer(obs, n):
    """The LLM proposes candidates. It does NOT get to pick the winner."""
    hist = ", ".join(f"({a:.2f} -> {r:.2f})" for a, r in sorted(obs))
    txt = llm_text(
        "You propose candidate parameter values for an optimizer. "
        "Reply with ONLY a JSON list of numbers. No prose.",
        f"Observed (launch_angle_deg -> range_m): {hist}\n\n"
        f"Propose {n} new angles strictly between 0 and 90 worth testing next to "
        f"maximize range. Reply with only a JSON list.",
        temperature=0.7)
    return [float(m.group()) for m in re.finditer(r"\d+(?:\.\d+)?", txt)][:n]


obs = mpc_optimize(llm_proposer, simulate_range)
best_angle, best_range = max(obs, key=lambda o: o[1])
print(f"\nbest: {best_angle:.2f} deg -> {best_range:.3f} m in {len(obs)} evaluations")

## 4.4 When to reach for MPC

**Use it when:**

- a wrong action is expensive or irreversible — burning budget, committing a mesh, running a physical experiment;
- you have a forward model cheaper than the action, and you trust it more than the LLM;
- actions compose, so the state you reach depends on the order you act in. That is where planning *N* steps and discarding all but the first genuinely differs from choosing one action at a time. The projectile here does not have this property, which is why the one-step form is enough.

**Don't bother when:**

- simulating a plan costs about what executing it costs — just execute and observe (§4.2);
- you have no forward model. Scoring candidates with the same LLM that proposed them adds cost without adding information;
- the problem is small and smooth, where `scipy.optimize.minimize_scalar` will beat this in both accuracy and evaluations.

## 4.5 Further reading

- **[Model Predictive Control](https://en.wikipedia.org/wiki/Model_predictive_control)** — the control-theory original, including the receding horizon this notebook simplifies away.
- **[Surrogate models](https://en.wikipedia.org/wiki/Surrogate_model)** — the general idea our quadratic fit is the simplest case of.
- **[Bayesian optimization](https://en.wikipedia.org/wiki/Bayesian_optimization)** — what this loop becomes when the surrogate also models its own uncertainty and you use that to choose where to sample.
- **[`scipy.optimize`](https://docs.scipy.org/doc/scipy/reference/optimize.html)** — the baseline worth comparing against before reaching for any of this.